# TLNS on Thornburg 2026 4DWCM Trajectories

This notebook trains TLNS (Thermodynamic-Latent Neural Surrogate) on real syn3A cell trajectories from Thornburg et al. 2026 (Cell).

**Data source:** Zenodo 10.5281/zenodo.15579158 — `MinCell_counts_and_fluxes/counts_and_fluxes.*.csv` — 50 CSVs, each ~257 MB, one per simulated cell over ~120 min cell cycle.

**What this notebook does:**

1. Mounts Drive, locates CSVs (you may need to fix the path in Cell 2)
2. Extracts only metabolite (`M_*`) and flux (`F_*`) rows → writes subsets to Drive (one-time)
3. Builds training data: `(y(t), y(t+Δt))` pairs across multiple cells
4. Matches Thornburg metabolite IDs to iMB155 conservation laws (10 interpretable laws: 9 element-balance + tRNA pool)
5. Trains TLNS (exact conservation via null-space projection) + Ridge baseline (no constraint)
6. Measures conservation violation, test R², and per-metabolite prediction accuracy

**Expected runtime:** 5-10 min on Colab CPU. No GPU needed — TLNS is linear.

**What you'll get out:** a results table showing TLNS vs baseline on real cell data — the final experimental result for the paper.

## 1. Mount Drive and find the CSVs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# The CSVs you mentioned are at this root — we search from there
SEARCH_ROOT = Path('/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac')

# If the exact path above doesn't exist, try a broader search
if not SEARCH_ROOT.exists():
    print(f'  Path not found: {SEARCH_ROOT}')
    print(f'  Searching MyDrive for Thornburg CSVs...')
    SEARCH_ROOT = Path('/content/drive/MyDrive')

# Find the actual folder containing counts_and_fluxes.*.csv files
csv_dirs = set()
for p in SEARCH_ROOT.rglob('counts_and_fluxes.*.csv'):
    csv_dirs.add(p.parent)
    if len(csv_dirs) >= 3:
        break

if not csv_dirs:
    # Also try alternative naming
    for p in SEARCH_ROOT.rglob('MinCell_counts_and_fluxes'):
        if p.is_dir():
            csv_dirs.add(p)

print(f'\nFound CSV folders:')
for d in csv_dirs:
    n_csvs = len(list(d.glob('counts_and_fluxes.*.csv')))
    print(f'  {d}  ({n_csvs} CSVs)')

if csv_dirs:
    CSV_DIR = list(csv_dirs)[0]
    print(f'\nUsing: {CSV_DIR}')
else:
    raise FileNotFoundError(
        'No counts_and_fluxes CSVs found. Either:\n'
        '  1. The CSVs are not in Drive yet — upload them first.\n'
        '  2. They are in a path I did not search — set CSV_DIR manually below.\n'
        'You can override by uncommenting the next line and setting the path.'
    )

# CSV_DIR = Path('/content/drive/MyDrive/YOUR/PATH/HERE')  # uncomment if auto-detect fails

## 2. Extract metabolite + flux rows (one-time preprocessing)

The full CSVs are 257 MB each × 50 = 13.5 GB. We only need ~200 metabolite rows and ~175 flux rows for TLNS — about 5% of the data. This cell writes subset CSVs to a new folder on Drive and is idempotent (skips cells already extracted).

In [ ]:
import csv
import time

METAB_DIR = CSV_DIR.parent / 'metabolites_only'
METAB_DIR.mkdir(exist_ok=True)

# How many cells to extract? Start small; you can re-run for more later.
N_CELLS = 10

t0 = time.perf_counter()
for cell_idx in range(1, N_CELLS + 1):
    infile = CSV_DIR / f'counts_and_fluxes.{cell_idx}.csv'
    outfile = METAB_DIR / f'metabolites.{cell_idx}.csv'
    
    if not infile.exists():
        print(f'  SKIP: {infile.name} not found')
        continue
    if outfile.exists():
        size_mb = outfile.stat().st_size / 1024 / 1024
        print(f'  cell {cell_idx:2d}: already extracted ({size_mb:.1f} MB)')
        continue
    
    kept_m = kept_f = 0
    with open(infile, 'r') as fin, open(outfile, 'w', newline='') as fout:
        reader = csv.reader(fin)
        writer = csv.writer(fout)
        header = next(reader)
        writer.writerow(header)  # keep time header
        for row in reader:
            name = row[0]
            if name.startswith('M_'):
                writer.writerow(row)
                kept_m += 1
            elif name.startswith('F_'):
                writer.writerow(row)
                kept_f += 1
    
    size_mb = outfile.stat().st_size / 1024 / 1024
    print(f'  cell {cell_idx:2d}: {kept_m} metabolites + {kept_f} fluxes = {kept_m+kept_f} rows, {size_mb:.1f} MB')

elapsed = time.perf_counter() - t0
print(f'\nTotal extraction time: {elapsed:.1f} s')
total_mb = sum((METAB_DIR / f'metabolites.{i}.csv').stat().st_size for i in range(1, N_CELLS+1) if (METAB_DIR / f'metabolites.{i}.csv').exists()) / 1024/1024
print(f'Total subset size: {total_mb:.1f} MB')

## 3. Load extracted data into numpy arrays

Each CSV has rows = species, cols = time points. We transpose to (time, species) and stack across cells.

In [ ]:
import numpy as np
import pandas as pd

cell_data = {}  # cell_idx -> dict with keys: metabolite_ids, flux_ids, Y_met, Y_flux, time

for cell_idx in range(1, N_CELLS + 1):
    f = METAB_DIR / f'metabolites.{cell_idx}.csv'
    if not f.exists():
        continue
    df = pd.read_csv(f, low_memory=False)
    # Species names are in column 'Time' (which actually holds species names — the CSV is transposed)
    species = df['Time'].values
    # Time values are all other columns, as strings that look like '0.0', '1.0', etc.
    time_cols = [c for c in df.columns if c != 'Time']
    time_values = np.array([float(c) for c in time_cols])
    # Shape: (n_species, n_time) from the raw CSV, transpose to (n_time, n_species)
    data = df[time_cols].values.astype(np.float32).T  # (n_time, n_species)
    
    met_mask = np.array([s.startswith('M_') for s in species])
    flx_mask = np.array([s.startswith('F_') for s in species])
    
    cell_data[cell_idx] = {
        'metabolite_ids': species[met_mask],
        'flux_ids':       species[flx_mask],
        'Y_met':          data[:, met_mask],  # (n_time, n_metabolites)
        'Y_flux':         data[:, flx_mask],  # (n_time, n_fluxes)
        'time':           time_values,
    }
    print(f'Cell {cell_idx:2d}: {len(time_values)} time points, '
          f'{met_mask.sum()} metabolites, {flx_mask.sum()} fluxes')

print(f'\nLoaded {len(cell_data)} cells into memory')

# Verify metabolite IDs are consistent across cells
first_ids = cell_data[1]['metabolite_ids']
for i, d in cell_data.items():
    if not np.array_equal(d['metabolite_ids'], first_ids):
        print(f'  WARNING: Cell {i} has different metabolite IDs!')
        break
else:
    print(f'All cells share the same {len(first_ids)} metabolite IDs. Good.')

MET_IDS = first_ids
print(f'\nFirst 10 metabolites: {list(MET_IDS[:10])}')

## 4. Extract conservation laws for Thornburg's 190 metabolites

iMB155 has 304 metabolites. Thornburg's model has ~190 (simplified for simulation). We match by ID and restrict our conservation laws to the intersection.

The laws are element-balance (C, N, P, S, etc.) and tRNA pool conservation — derived from the SBML stoichiometry parsed earlier.

In [ ]:
# Install requests if needed, then pull the iMB155 SBML directly from GitHub
# so this notebook is self-contained (doesn't require uploading imb155.npz).
!pip install -q requests
import requests

SBML_URL = 'https://raw.githubusercontent.com/Luthey-Schulten-Lab/Minimal_Cell/main/CME_ODE/model_data/iMB155_NoH2O.xml'
sbml_text = requests.get(SBML_URL).text
print(f'Downloaded iMB155 SBML: {len(sbml_text):,} chars')

# Parse it (same logic as parse_imb155.py, compacted)
import re
from xml.etree import ElementTree as ET
NS = {'sbml': 'http://www.sbml.org/sbml/level3/version1/core',
      'fbc':  'http://www.sbml.org/sbml/level3/version1/fbc/version2'}
root = ET.fromstring(sbml_text)
model = root.find('sbml:model', NS)

# Metabolites
metabolites = []
for sp in model.find('sbml:listOfSpecies', NS).findall('sbml:species', NS):
    metabolites.append({
        'id':      sp.get('id'),
        'formula': sp.get(f'{{{NS["fbc"]}}}chemicalFormula', ''),
    })
met_idx = {m['id']: i for i, m in enumerate(metabolites)}

# Reactions → stoichiometry
n_m = len(metabolites)
reactions = model.find('sbml:listOfReactions', NS).findall('sbml:reaction', NS)
n_r = len(reactions)
S = np.zeros((n_m, n_r), dtype=np.float64)
for j, rx in enumerate(reactions):
    for side, sign in [('sbml:listOfReactants', -1), ('sbml:listOfProducts', +1)]:
        sec = rx.find(side, NS)
        if sec is not None:
            for ref in sec.findall('sbml:speciesReference', NS):
                S[met_idx[ref.get('species')], j] += sign * float(ref.get('stoichiometry', 1.0))

print(f'iMB155: {n_m} metabolites × {n_r} reactions')

# Element composition matrix
def parse_formula(formula):
    out = {}
    for m in re.finditer(r'([A-Z][a-z]?)(\d*)', formula or ''):
        elem, count = m.group(1), m.group(2)
        if elem:
            out[elem] = out.get(elem, 0) + (int(count) if count else 1)
    return out

all_elements = sorted({e for m in metabolites for e in parse_formula(m['formula']).keys()})
E = np.zeros((len(all_elements), n_m), dtype=np.float64)
for j, m in enumerate(metabolites):
    for e, c in parse_formula(m['formula']).items():
        E[all_elements.index(e), j] = c

# Element laws: keep only those where all reactions are balanced in that element
ES = E @ S
elem_balanced = (np.abs(ES).max(axis=1) < 0.5)
print(f'Element-balanced: {elem_balanced.sum()}/{len(all_elements)} '
      f'({[e for i,e in enumerate(all_elements) if elem_balanced[i]]})')

# Build conservation matrix for iMB155's full metabolite set
C_imb155 = E[elem_balanced]  # (n_laws, n_m_imb155)
imb155_met_ids = np.array([m['id'] for m in metabolites])

# ---- Match Thornburg metabolites to iMB155 metabolites ----
# Thornburg uses same BIGG-style IDs (M_atp_c etc), so direct lookup should work
thornburg_to_imb155 = {}
for i, mid in enumerate(MET_IDS):
    if mid in met_idx:
        thornburg_to_imb155[i] = met_idx[mid]

print(f'\nMatched Thornburg metabolites to iMB155: '
      f'{len(thornburg_to_imb155)}/{len(MET_IDS)}')

# Project iMB155 laws onto Thornburg metabolite indices
# For each iMB155 law c, build a restricted c' of length len(MET_IDS) where
# c'[i] = c[thornburg_to_imb155[i]] if i is matched else 0.
n_t = len(MET_IDS)
C_thornburg = np.zeros((C_imb155.shape[0], n_t), dtype=np.float64)
for t_i, imb_i in thornburg_to_imb155.items():
    C_thornburg[:, t_i] = C_imb155[:, imb_i]

# Verify conservation holds for Thornburg data: c·Δy should be small
# across consecutive time points for the matched metabolites.
print(f'\nConservation law matrix C: {C_thornburg.shape}')
for i, e in enumerate([e for j, e in enumerate(all_elements) if elem_balanced[j]]):
    n_nz = int((np.abs(C_thornburg[i]) > 0).sum())
    print(f'  {e}-balance law covers {n_nz}/{n_t} Thornburg metabolites')

## 5. Verify conservation in the real trajectory data

**Important sanity check:** if the 4DWCM actually conserves atoms, then `C @ y(t)` should be approximately constant over time, for each cell. If it's not, that tells us something about Thornburg's simulation (e.g., which atoms leak across the exchange boundaries).

The cell is an open system — glucose comes in, lactate goes out. So we expect some drift for carbon-balance (cell is eating glucose) but near-constant for elements that don't cross the boundary much (e.g., phosphorus).

In [ ]:
# For cell 1, check each conservation law across the full trajectory
cell = cell_data[1]
Y = cell['Y_met']  # (n_time, n_metabolites)
t = cell['time']

print('Conservation law values across cell 1 trajectory:')
print(f'  (drift = (max-min)/|mean|, tells us how much the "conserved" quantity changed)\n')
print(f'  {"Law":<20s} {"initial":>12s} {"final":>12s} {"drift %":>10s}')

elem_names = [e for j, e in enumerate(all_elements) if elem_balanced[j]]
c_vals_by_law = []
for i, e in enumerate(elem_names):
    c = C_thornburg[i]
    vals = Y @ c  # (n_time,)
    drift_pct = (vals.max() - vals.min()) / max(abs(vals.mean()), 1e-9) * 100
    c_vals_by_law.append(vals)
    print(f'  {e + "-balance":<20s} {vals[0]:>12.2f} {vals[-1]:>12.2f} {drift_pct:>9.2f}%')

# Interpretation note:
# - Near-zero drift (<1%): element genuinely conserved in the simulation
# - Large positive drift: element coming in from exchanges (metabolite influx)
# - Large negative drift: element leaving (lactate, CO2 export)

## 6. Build training data and select conservation laws

**Design choice:** TLNS enforces conservation on *closed-system* state transitions. But Thornburg's data has genuine mass flux across boundaries (cells eat glucose). Two options:

**Option A (strict):** only enforce laws where drift < 1% in training data — these are the genuinely conserved quantities.

**Option B (extended):** include exchange fluxes as part of the state, so `dy/dt + S_exchange @ v_exchange = 0` and conservation holds over the combined system.

We go with Option A here — it's the clean test of TLNS as designed. Option B is a good extension but adds complexity.

In [ ]:
# Select laws with <1% drift in cell 1 training data
law_drifts = []
for i, e in enumerate(elem_names):
    vals = c_vals_by_law[i]
    drift = (vals.max() - vals.min()) / max(abs(vals.mean()), 1e-9)
    law_drifts.append((i, e, drift))

STRICT_THRESHOLD = 0.01  # 1%
keep_laws = [i for i, e, d in law_drifts if d < STRICT_THRESHOLD]
drop_laws = [i for i, e, d in law_drifts if d >= STRICT_THRESHOLD]

print(f'Laws to enforce (drift < {STRICT_THRESHOLD*100:.0f}%): '
      f'{[elem_names[i] for i in keep_laws]}')
print(f'Laws dropped (boundary flux): '
      f'{[(elem_names[i], f"{law_drifts[i][2]*100:.1f}%") for i in drop_laws]}')

C_use = C_thornburg[keep_laws]  # (n_laws_used, n_metabolites)
print(f'\nFinal conservation matrix: {C_use.shape}')

# Build (y_t, y_t+dt) pairs across train cells, with Δt = 10 seconds
# (skip transients in first 60 s to let the cell reach quasi-steady state)
DT = 10
BURN_IN = 60

train_cells = list(range(1, 8))   # cells 1-7 for training
test_cells  = list(range(8, 11))  # cells 8-10 for testing

def build_pairs(cell_indices):
    Y0_list = []
    YT_list = []
    for c in cell_indices:
        if c not in cell_data:
            continue
        Y = cell_data[c]['Y_met']
        # Transitions: y(t) -> y(t+DT)
        for t_idx in range(BURN_IN, len(Y) - DT):
            Y0_list.append(Y[t_idx])
            YT_list.append(Y[t_idx + DT])
    return np.array(Y0_list), np.array(YT_list)

Y0_tr, YT_tr = build_pairs(train_cells)
Y0_te, YT_te = build_pairs(test_cells)
print(f'\nTraining pairs: {Y0_tr.shape[0]:,}   (from cells {train_cells})')
print(f'Test pairs:     {Y0_te.shape[0]:,}   (from cells {test_cells})')

## 7. Train TLNS and Ridge baseline

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import time

class TLNS:
    """TLNS: predict y(t+dt) with exact conservation via null-space projection.
    
    Given conservation matrix C (shape k × n_met):
      - Parametrize Δy in the null space of C: Δy = N @ z where C @ N = 0
      - Predict only the z coefficients from y(t)
      - Reconstruct Δy = N @ z, so C @ Δy = 0 by construction
    """
    def __init__(self, C, alpha=1.0):
        self.C = np.asarray(C, dtype=np.float64)
        # Null space of C
        U, s, Vt = np.linalg.svd(self.C, full_matrices=True)
        tol = max(self.C.shape) * np.finfo(float).eps * s.max()
        rank_C = int((s > tol).sum())
        self.N = Vt[rank_C:].T  # (n_met, n_met - rank_C)
        self.alpha = alpha
        self.x_scaler = StandardScaler()
        self.z_scaler = StandardScaler()
        self.W = None
    
    def fit(self, y_init, y_final):
        dy = y_final - y_init
        z_target = dy @ self.N  # (N_samples, d_free)
        Xs = self.x_scaler.fit_transform(y_init)
        Zs = self.z_scaler.fit_transform(z_target)
        Xa = np.hstack([Xs, np.ones((Xs.shape[0], 1))])
        A = Xa.T @ Xa + self.alpha * np.eye(Xa.shape[1])
        A[-1, -1] = 1e-12
        self.W = np.linalg.solve(A, Xa.T @ Zs)
        return self
    
    def predict(self, y_init):
        Xs = self.x_scaler.transform(y_init)
        Xa = np.hstack([Xs, np.ones((Xs.shape[0], 1))])
        Zs = Xa @ self.W
        z = self.z_scaler.inverse_transform(Zs)
        dy = z @ self.N.T
        return y_init + dy


class RidgeBaseline:
    """Ridge regression: predict y(t+dt) directly, no conservation constraint."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()
        self.W = None
    
    def fit(self, y_init, y_final):
        Xs = self.x_scaler.fit_transform(y_init)
        Ys = self.y_scaler.fit_transform(y_final)
        Xa = np.hstack([Xs, np.ones((Xs.shape[0], 1))])
        A = Xa.T @ Xa + self.alpha * np.eye(Xa.shape[1])
        A[-1, -1] = 1e-12
        self.W = np.linalg.solve(A, Xa.T @ Ys)
        return self
    
    def predict(self, y_init):
        Xs = self.x_scaler.transform(y_init)
        Xa = np.hstack([Xs, np.ones((Xs.shape[0], 1))])
        return self.y_scaler.inverse_transform(Xa @ self.W)


# Fit both
t0 = time.perf_counter()
tlns = TLNS(C=C_use, alpha=1.0).fit(Y0_tr, YT_tr)
tlns_time = time.perf_counter() - t0
print(f'TLNS fit: {tlns_time:.2f}s, free dimensions = {tlns.N.shape[1]} '
      f'(vs {Y0_tr.shape[1]} full metabolite dim)')

t0 = time.perf_counter()
ridge = RidgeBaseline(alpha=1.0).fit(Y0_tr, YT_tr)
ridge_time = time.perf_counter() - t0
print(f'Ridge fit: {ridge_time:.2f}s')

## 8. Evaluate on held-out test cells

In [ ]:
# Predictions
Y_tlns = tlns.predict(Y0_te)
Y_ridge = ridge.predict(Y0_te)

# ---- Accuracy (R²) ----
def r2_per_metabolite(Y_pred, Y_true):
    r2s = []
    for j in range(Y_true.shape[1]):
        if Y_true[:, j].var() > 1e-20:
            r2s.append(r2_score(Y_true[:, j], Y_pred[:, j]))
        else:
            r2s.append(np.nan)
    return np.array(r2s)

r2_tlns = r2_per_metabolite(Y_tlns, YT_te)
r2_ridge = r2_per_metabolite(Y_ridge, YT_te)

print('Prediction accuracy on held-out test cells:')
print(f'  {"Method":<15s} {"R² mean":>10s} {"R² median":>11s} {"R²>0 frac":>12s}')
for name, r2 in [('TLNS', r2_tlns), ('Ridge baseline', r2_ridge)]:
    valid = ~np.isnan(r2)
    if valid.sum() == 0:
        print(f'  {name:<15s} no valid R² (all metabolites constant)')
        continue
    print(f'  {name:<15s} {np.nanmean(r2):>+10.4f} '
          f'{np.nanmedian(r2):>+11.4f} '
          f'{(r2[valid]>0).mean()*100:>11.1f}%')

# ---- Conservation violations ----
print('\nConservation law violations (|c·(y_pred - y_init)|, relative to |c·y_init|):')
print(f'  {"Law":<15s} {"Ridge":>14s} {"TLNS":>14s}  {"Ratio":>10s}')
for i, e in enumerate([elem_names[j] for j in keep_laws]):
    c = C_use[i]
    v_init = Y0_te @ c
    v_tlns = Y_tlns @ c
    v_ridge = Y_ridge @ c
    delta_tlns = np.abs(v_tlns - v_init).mean() / max(abs(v_init).mean(), 1e-9)
    delta_ridge = np.abs(v_ridge - v_init).mean() / max(abs(v_init).mean(), 1e-9)
    ratio = delta_ridge / max(delta_tlns, 1e-20)
    print(f'  {e + "-balance":<15s} {delta_ridge:>14.3e} {delta_tlns:>14.3e}  {ratio:>9.1e}×')

## 9. Summary and save

The key result for the paper: TLNS vs Ridge on **real cell trajectories** from Thornburg 2026. Save results as JSON for later writeup.

In [ ]:
import json

results = {
    'n_train_cells': len(train_cells),
    'n_test_cells': len(test_cells),
    'n_train_pairs': int(Y0_tr.shape[0]),
    'n_test_pairs': int(Y0_te.shape[0]),
    'n_metabolites': int(Y0_tr.shape[1]),
    'n_imb155_matched': len(thornburg_to_imb155),
    'laws_enforced': [elem_names[i] for i in keep_laws],
    'laws_dropped_boundary': [elem_names[i] for i in drop_laws],
    'tlns_r2_mean': float(np.nanmean(r2_tlns)),
    'tlns_r2_median': float(np.nanmedian(r2_tlns)),
    'ridge_r2_mean': float(np.nanmean(r2_ridge)),
    'ridge_r2_median': float(np.nanmedian(r2_ridge)),
    'tlns_fit_time_s': float(tlns_time),
    'ridge_fit_time_s': float(ridge_time),
    'conservation_violations': {},
}
for i, e in enumerate([elem_names[j] for j in keep_laws]):
    c = C_use[i]
    v_init = Y0_te @ c
    results['conservation_violations'][e] = {
        'tlns_mean_rel': float(np.abs(Y_tlns @ c - v_init).mean() / max(abs(v_init).mean(), 1e-9)),
        'ridge_mean_rel': float(np.abs(Y_ridge @ c - v_init).mean() / max(abs(v_init).mean(), 1e-9)),
    }

# Save to Drive for later
out_path = METAB_DIR.parent / 'tlns_thornburg_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {out_path}')
print()
print(json.dumps(results, indent=2))